In [1]:
import json
import os
import time

from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from groq import Groq

In [2]:
PROJECT_ROOT = Path.cwd().parent

ANALYSES_PATH = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "paper_analyses.csv"
)

QUERY_CONFIG_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "arxiv_query.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_PATH = OUTPUT_DIR / "reviewed_papers.csv"

MODEL = "llama-3.1-8b-instant"

In [3]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

In [4]:
with open(
    QUERY_CONFIG_PATH,
    encoding="utf-8",
) as file:
    query_config = json.load(file)

research_topic = query_config["topic"]

print(research_topic)

i want to investigate about the gaussian distribution


In [5]:
df_analyses = pd.read_csv(ANALYSES_PATH)

In [11]:
reviews = []

for _, paper in df_analyses.iterrows():

    prompt = f"""
You are the Reviewer Agent in an academic research system.

The user's research topic is:

{research_topic}

Your task is to review an analysis created by another AI agent.

Original paper title:
{paper["title"]}

Original abstract:
{paper["original_abstract"]}

Analysis produced by the Analyst Agent:

Summary:
{paper["summary"]}

Main problem:
{paper["main_problem"]}

Main contribution:
{paper["main_contribution"]}

Applications:
{paper["applications"]}

Limitations:
{paper["limitations"]}

Original relevance score:
{paper["relevance_score"]}

Original relevance reason:
{paper["relevance_reason"]}

Return only a valid JSON object with exactly these fields:

{{
  "approved": true or false,
  "review_comments": "Short explanation of the review",
  "unsupported_claims": "Claims not supported by the abstract, or None",
  "corrected_relevance_score": integer from 1 to 10,
  "final_relevance_reason": "Final explanation of the paper's relevance"
}}

Relevance scoring criteria:

- 9-10: Directly addresses the user's research topic.
- 7-8: Strongly related, but not a direct match.
- 5-6: Partially related or useful only for background.
- 3-4: Weakly related.
- 1-2: Essentially unrelated.

Rules:

- Evaluate each paper independently.
- Do not default to the original score.
- Keep the original relevance score if it is well justified.
- Change it only when the abstract supports a different score.
- Use the full relevance scale from 1 to 10.
- Do not assign the same score automatically to different papers.
- approved must be true if the analysis is broadly accurate and supported.
- approved must be false if the analysis contains important unsupported claims.
- corrected_relevance_score must be an integer between 1 and 10.
- Use only the information present in the abstract.
- Return only valid JSON.
- Do not use Markdown.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={
            "type": "json_object"
        },
        temperature=0.1,
    )

    raw_review = response.choices[0].message.content

    review = json.loads(raw_review)

    review["title"] = paper["title"]
    review["authors"] = paper["authors"]
    review["published"] = paper["published"]
    review["url"] = paper["url"]
    review["original_abstract"] = paper["original_abstract"]

    review["analyst_summary"] = paper["summary"]
    review["main_problem"] = paper["main_problem"]
    review["main_contribution"] = paper["main_contribution"]
    review["applications"] = paper["applications"]
    review["limitations"] = paper["limitations"]

    review["original_relevance_score"] = paper["relevance_score"]

    reviews.append(review)

    print(f'Reviewed: {paper["title"][:70]}...')

    time.sleep(1)

Reviewed: On Polymer Statistical Mechanics: From Gaussian Distribution to Maxwel...
Reviewed: A Constructive Approach to $q$-Gaussian Distributions: $α$-Divergence ...
Reviewed: Revisiting De Moivre-Laplace...
Reviewed: Gaussian Approximation of Convex Sets by Intersections of Halfspaces...
Reviewed: The fast rate of convergence of the smooth adapted Wasserstein distanc...


In [12]:
df_reviews = pd.DataFrame(reviews)

In [13]:
df_reviews["approved"].value_counts()

approved
True    5
Name: count, dtype: int64

In [14]:
df_reviews

,approved,review_comments,unsupported_claims,corrected_relevance_score,final_relevance_reason,title,authors,published,url,original_abstract,analyst_summary,main_problem,main_contribution,applications,limitations,original_relevance_score
0,True,The analysis is broadly accurate and supported...,None,8,The paper is strongly related to the user's to...,On Polymer Statistical Mechanics: From Gaussia...,Lixiang Yang,2023-08-22T14:54:57Z,http://arxiv.org/abs/2308.11482v1,Macroscopic mechanical properties of polymers ...,This paper discusses the application of probab...,The insufficiency of the Gaussian distribution...,The proposal of using Maxwell-Boltzmann and Fe...,Understanding polymer mechanics problems such ...,The abstract does not provide information on t...,9
1,True,The analysis is broadly accurate and supported...,None,9,The paper directly addresses the user's resear...,A Constructive Approach to $q$-Gaussian Distri...,"Hiroki Suyari, Antonio M. Scarfone",2026-03-22T20:25:23Z,http://arxiv.org/abs/2603.21391v2,The Large Deviation Principle (LDP) and the Ce...,The paper presents a constructive approach to ...,Deriving a constructive probabilistic framewor...,Establishing a generalized binomial distributi...,The paper's framework and results can be appli...,The paper's results are limited to the regime ...,8
2,True,The analysis is broadly accurate and supported...,None,8,The paper is strongly related to the user's to...,Revisiting De Moivre-Laplace,Raphaël Cerf,2025-12-26T16:28:57Z,http://arxiv.org/abs/2512.22330v1,We revisit the proof of the de Moivre--Laplace...,The paper revisits the proof of the de Moivre-...,The main problem addressed by the paper is to ...,The main contribution of the paper is a new pr...,The paper's contribution could have potential ...,The paper's focus on undergraduate education a...,7
3,True,The analysis accurately summarizes the paper's...,None,8,The paper is strongly related to the user's to...,Gaussian Approximation of Convex Sets by Inter...,"Anindya De, Shivam Nadimpalli, Rocco A. Servedio",2023-11-14T22:42:47Z,http://arxiv.org/abs/2311.08575v1,We study the approximability of general convex...,The paper studies the approximability of conve...,Approximating convex sets by intersections of ...,Establishing upper and lower bounds for the nu...,Potential applications in fields such as machi...,The paper focuses on convex sets and intersect...,7
4,True,The analysis is broadly accurate and supported...,None,8,The paper is strongly related to the user's to...,The fast rate of convergence of the smooth ada...,"Martin Larsson, Jonghwa Park, Johannes Wiesel",2025-03-13T19:16:19Z,http://arxiv.org/abs/2503.10827v2,Estimating a $d$-dimensional distribution $μ$ ...,The paper investigates the convergence rate of...,The curse of dimensionality in estimating a di...,The paper shows that the smooth adapted Wasser...,The results of this paper could be applied to ...,The paper assumes that the underlying measure ...,6


In [9]:
df_reviews.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Reviews saved to:\n{OUTPUT_PATH}")

Reviews saved to:
C:\Users\jortialo\ai-paper-review-agentic\data\intermediate\reviewed_papers.csv
